In [1]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

In [2]:
from src.pipeline.config import (
	TRAIN_TRANSACTION_PATH,
	TRAIN_IDENTITY_PATH,
	PROCESSED_DATA_DIR,
	RANDOM_SEED
)

In [3]:
df_trans = pd.read_csv(TRAIN_TRANSACTION_PATH, engine="pyarrow")
df_id = pd.read_csv(TRAIN_IDENTITY_PATH, engine="pyarrow")

print(f"Transaction shape: {df_trans.shape}")
print(f"Identity shape: {df_id.shape}")

Transaction shape: (590540, 394)
Identity shape: (144233, 41)


In [4]:
df = pd.merge(df_trans, df_id, on="TransactionID", how="left")
print(f"Final df shape: {df.shape}")

Final df shape: (590540, 434)


In [5]:
processed_path = PROCESSED_DATA_DIR / "train_merged.parquet"
df.to_parquet(processed_path, engine="pyarrow")
print(f"Saved in {processed_path}")

Saved in /home/arcsin/IEEE_CIS/data/processed/train_merged.parquet


In [6]:
df.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [7]:
print(f"RAM usage: {df.memory_usage().sum() / 1024**2:2f} MB")
print(df.dtypes.value_counts())

RAM usage: 1984.064550 MB
float64    399
str         31
int64        4
Name: count, dtype: int64


In [8]:
df[["TransactionDT", "TransactionAmt"]].describe()

,TransactionDT,TransactionAmt
count,5.905400e+05,590540.000000
mean,7.372311e+06,135.027176
std,4.617224e+06,239.162522
min,8.640000e+04,0.251000
25%,3.027058e+06,43.321000
50%,7.306528e+06,68.769000
75%,1.124662e+07,125.000000
max,1.581113e+07,31937.391000


In [9]:
print(df['isFraud'].value_counts(normalize=True) * 100)

isFraud
0    96.500999
1     3.499001
Name: proportion, dtype: float64


In [10]:
X = df.drop(["isFraud", "DeviceInfo", "TransactionID"], axis=1)
y = df["isFraud"]

In [11]:
kaggle_cat_cols = (
    ['ProductCD'] +
    [f'card{i}' for i in range(1, 7)] +
    ['addr1', 'addr2'] +
    ['P_emaildomain', 'R_emaildomain'] +
    [f'M{i}' for i in range(1, 10)] +
    ['DeviceType', 'DeviceInfo'] +
    [f'id_{i}' for i in range(12, 39)]
)

cat_features = [col for col in kaggle_cat_cols if col in X.columns]

for col in cat_features:
    X[col] = X[col].fillna('missing').astype(str)

print(X[cat_features].nunique().sort_values(ascending=False))

card1            13553
id_19              523
card2              501
id_21              491
id_20              395
id_25              342
addr1              333
id_33              261
id_31              131
card5              120
card3              115
id_17              105
id_26               96
id_30               76
addr2               75
R_emaildomain       61
P_emaildomain       60
id_13               55
id_22               26
id_14               26
id_18               19
id_24               13
card4                5
card6                5
id_32                5
ProductCD            5
id_34                5
id_23                4
M4                   4
id_15                4
M2                   3
M1                   3
M5                   3
M3                   3
M6                   3
M8                   3
M9                   3
DeviceType           3
id_12                3
M7                   3
id_16                3
id_29                3
id_27                3
id_28      

In [12]:
cat_features.remove('card1')
len(cat_features)

47

In [13]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
	X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)

In [14]:
from catboost import CatBoostClassifier

clf = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    random_seed=RANDOM_SEED,
    eval_metric='AUC',
    custom_metric=['Logloss'],
	task_type='GPU',
    verbose=10
)

In [15]:
import gc
gc.collect()

clf.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_test, y_test),
    use_best_model=True
)

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7050193	best: 0.7050193 (0)	total: 226ms	remaining: 1m 7s
10:	test: 0.8305402	best: 0.8305402 (10)	total: 1.63s	remaining: 42.9s
20:	test: 0.8551967	best: 0.8551967 (20)	total: 3.07s	remaining: 40.8s
30:	test: 0.8886856	best: 0.8886856 (30)	total: 4.43s	remaining: 38.4s
40:	test: 0.8953375	best: 0.8953375 (40)	total: 5.84s	remaining: 36.9s
50:	test: 0.9001637	best: 0.9001637 (50)	total: 7.18s	remaining: 35.1s
60:	test: 0.9040974	best: 0.9040974 (60)	total: 8.52s	remaining: 33.4s
70:	test: 0.9060127	best: 0.9060127 (70)	total: 9.8s	remaining: 31.6s
80:	test: 0.9101653	best: 0.9101653 (80)	total: 11.1s	remaining: 30.1s
90:	test: 0.9153837	best: 0.9153837 (90)	total: 12.5s	remaining: 28.6s
100:	test: 0.9182929	best: 0.9182929 (100)	total: 13.9s	remaining: 27.3s
110:	test: 0.9200697	best: 0.9200697 (110)	total: 15.2s	remaining: 26s
120:	test: 0.9235967	best: 0.9235967 (120)	total: 16.6s	remaining: 24.6s
130:	test: 0.9250214	best: 0.9250214 (130)	total: 18.1s	remaining: 23.3s
140

CatBoostClassifier(custom_metric=['Logloss'], eval_metric='AUC', iterations=300, learning_rate=0.1, random_seed=42, task_type='GPU', verbose=10)

In [16]:
from sklearn.metrics import roc_auc_score

val_preds = clf.predict_proba(X_test)[:, 1]

auc_score = roc_auc_score(y_test, val_preds)
print(f"Baseline ROC-AUC: {auc_score:.5f}")

Baseline ROC-AUC: 0.93630


In [17]:
feature_imp = pd.Series(clf.get_feature_importance(), index=X_train.columns)
print(feature_imp.sort_values(ascending=False).head(10))

card2             6.455894
C13               5.458218
C14               4.196863
C1                4.196050
M5                4.131151
TransactionAmt    3.430517
P_emaildomain     3.263242
V308              2.689300
TransactionDT     2.543683
M4                2.323264
dtype: float64
